# Run the pruning experiments (Colab, T4 GPU)

Runtime → Change runtime type → **T4 GPU**, then run all cells.

You need: a GitHub token (read access to this repo) and a Hugging Face token from an account that has accepted the **Llama 3.2** license on huggingface.co.

Results are written to Google Drive (`MyDrive/reasoning-aware-llm-pruning-results`) after every run, so if Colab disconnects you can re-run all cells and finished runs are skipped. The full grid takes roughly 3 hours.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import getpass, os
token = getpass.getpass('GitHub token: ')
!git clone -q https://{token}@github.com/Arn15/reasoning-aware-llm-pruning.git 2>/dev/null || (cd reasoning-aware-llm-pruning && git pull -q)
%cd /content/reasoning-aware-llm-pruning
del token

In [ ]:
# keep results on Drive so they survive disconnects
DRIVE = '/content/drive/MyDrive/reasoning-aware-llm-pruning-results'
os.makedirs(DRIVE, exist_ok=True)
!rm -rf results && ln -s "$DRIVE" results
!ls results

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from huggingface_hub import login
login(token=getpass.getpass('Hugging Face token: '))

In [ ]:
# quick check (~2 min): perplexity only for the dense model
!python -m src.run --method dense --skip-bench --force && rm -f results/runs/*dense*.json

In [ ]:
# full grid (resumable)
!bash run_all.sh

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/results.md').read()))
display(Image('results/sparsity_curves.png'))
import glob
for p in glob.glob('results/sensitivity_*.png'): display(Image(p))